# Perfiles de interacción del cursor: clustering

**Pregunta de la propuesta.** ¿Qué perfiles de interacción del cursor aparecen en COMPASS?

**Qué tipo de modelo es.** El clustering es aprendizaje **no supervisado** (clase 1): el algoritmo recibe solo variables X y aprende una partición de las personas en grupos. No hay variable Y, así que no existe un "accuracy" ni un error de predicción. Lo que se evalúa es si los grupos son **reales**:

| Pregunta | Cómo se responde |
|---|---|
| ¿Los grupos están separados? | Silhouette |
| ¿Esa separación supera lo que produce un solo grupo sin estructura? | Comparación con datos simulados sin grupos (hipótesis nula) |
| ¿Los grupos se repiten en personas nuevas? | **Train/test**: formar los grupos con una parte de las personas y asignar al resto |
| ¿Y con otros nodos u otras muestras? | Estabilidad con remuestreo (bootstrap) y con mitades de nodos |
| ¿Los grupos significan algo fuera del modelo? | Comparar el MLQ externo entre grupos |

**Unidad de análisis:** la persona (n = 60).

**Dos formas de agrupar** (se comparan):
1. **Cómo deciden:** telemetría del cursor (tiempo, cambios, exploración).
2. **Qué deciden:** las alternativas A/B/C elegidas.

**Alternativa al clustering:** si la conducta forma un continuo y no grupos, un índice continuo (PCA) describe mejor a las personas. Se evalúa al final.

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.stats.multitest import multipletests

from sklearn.cluster import AgglomerativeClustering, KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.precision", 3)
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
sns.set_theme(style="whitegrid")

SEED = 42
DATA = Path("../data") if Path("../data").exists() else Path("data")
OUT = Path("output")

DIMS = ["iia", "iic", "mi", "ei", "ci", "rc", "dpe_a", "dpe_p", "lf"]
COMPUESTOS = {
    "transformacional": ["iia", "iic", "mi", "ei", "ci"],
    "transaccional": ["rc", "dpe_a"],
    "pasivo": ["dpe_p", "lf"],
}

## 0. Carga y construcción de variables

Son las mismas variables del notebook 01 (este notebook es autocontenido):

- **Telemetría por persona:** cada métrica se estandariza *dentro de cada nodo* (z-score entre las 60 personas) y luego se promedia por persona. Un valor positivo significa "más que el resto en los mismos nodos". Se usa solo X, nunca el MLQ.
- **Decisiones:** la alternativa elegida (A/B/C) en cada nodo.
- **MLQ externo:** solo se usa para **validar** los grupos al final, nunca para formarlos.

In [ ]:
ext = pd.read_csv(DATA / "puntajes_ext.csv")
compass = pd.read_csv(DATA / "puntajes_compass.csv")
decisiones = pd.read_csv(DATA / "explicit_decisions.csv", usecols=["session_id", "user_id", "node_id", "option_id"])
opciones = pd.read_csv(DATA / "option_process_stats.csv")
logs = pd.read_csv(DATA / "process_logs.csv",
                   usecols=["session_id", "user_id", "node_id", "node_total_duration_ms", "hover_switch_count",
                            "selected_option_was_most_hovered", "first_hover_option_id", "final_choice"])

# MLQ externo y compuestos (solo para validación externa)
Y = ext.set_index("user_id")[DIMS].copy()
for nombre, cols in COMPUESTOS.items():
    Y[nombre] = Y[cols].mean(axis=1)
OBJ_MLQ = list(COMPUESTOS) + DIMS

# Decisiones COMPASS (promedio por decisión)
X_dec = compass.set_index("user_id")[DIMS].div(compass.set_index("user_id")["n_nodos"], axis=0)
X_dec.columns = [f"dec_{c}" for c in DIMS]

In [ ]:
# Telemetría a nivel nodo (process_logs repite los agregados del nodo en cada evento)
nodo = logs.drop_duplicates(["session_id", "node_id"]).copy()
assert len(nodo) == len(decisiones)

op = opciones.merge(nodo[["session_id", "node_id", "final_choice"]], on=["session_id", "node_id"])
op["elegida"] = op["option_id"] == op["final_choice"]
por_nodo_op = op.groupby(["session_id", "node_id"]).agg(
    n_exploradas=("option_id", "nunique"),
    revisitas=("hover_count", "mean"),
    hover_total=("hover_total_ms", "sum"),
    hover_elegida=("hover_total_ms", lambda s: s[op.loc[s.index, "elegida"]].sum()),
).reset_index()
por_nodo_op["prop_hover_elegida"] = por_nodo_op["hover_elegida"] / por_nodo_op["hover_total"]

nodo = nodo.merge(por_nodo_op, on=["session_id", "node_id"], how="left")
nodo["log_duracion"] = np.log(nodo["node_total_duration_ms"] / 1000)
nodo["cambios"] = nodo["hover_switch_count"]
nodo["eligio_mas_vista"] = (nodo["selected_option_was_most_hovered"] == "t").astype(float)
nodo["eligio_primera"] = (nodo["first_hover_option_id"] == nodo["final_choice"]).astype(float)

METRICAS = ["log_duracion", "cambios", "n_exploradas", "revisitas", "prop_hover_elegida",
            "eligio_mas_vista", "eligio_primera"]

def z_en_nodo(s):
    sd = s.std(ddof=0)
    return (s - s.mean()) / sd if sd > 0 else s * 0.0

z = nodo[["user_id", "node_id"]].copy()
for m in METRICAS:
    z[m] = nodo.groupby("node_id")[m].transform(z_en_nodo)

# Tensor persona × nodo × métrica (NaN si la persona no recorrió el nodo).
# Permite recalcular los indicadores con cualquier subconjunto de nodos (estabilidad).
usuarios = sorted(nodo["user_id"].unique())
nodos = sorted(nodo["node_id"].unique())
T = np.full((len(usuarios), len(nodos), len(METRICAS)), np.nan)
iu = {u: i for i, u in enumerate(usuarios)}
inod = {n: i for i, n in enumerate(nodos)}
T[z["user_id"].map(iu).values, z["node_id"].map(inod).values, :] = z[METRICAS].values
print(f"{len(usuarios)} personas × {len(nodos)} nodos × {len(METRICAS)} métricas | celdas sin dato: {np.isnan(T[:, :, 0]).sum()}")

def features_tel(T, idx=None):
    """Indicadores por persona, usando solo los nodos de `idx` (todos si es None)."""
    sub = T if idx is None else T[:, idx, :]
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        media = np.nanmean(sub, axis=1)
        sd = np.nanstd(sub[:, :, METRICAS.index("log_duracion")], axis=1, ddof=1)
    F = pd.DataFrame(media, index=usuarios, columns=[f"tel_{c}" for c in METRICAS])
    F["tel_sd_log_duracion"] = sd
    return F

F_ALL = features_tel(T)
assert not F_ALL.isna().any().any()

# Dos conjuntos de indicadores del cursor
COLS_PROP = ["tel_log_duracion", "tel_cambios", "tel_n_exploradas"]   # los tres de la propuesta
COLS_AMPLIA = list(F_ALL.columns)                                     # los 8 disponibles
F_ALL[COLS_AMPLIA].describe().T[["mean", "std", "min", "max"]]

In [ ]:
# Decisiones A/B/C: solo nodos que las 60 personas recorrieron (excluye 20, 24, 27 y 28)
M = decisiones.pivot(index="user_id", columns="node_id", values="option_id").loc[usuarios]
nodos_completos = M.columns[M.notna().all()]
print(f"Nodos recorridos por las 60 personas: {len(nodos_completos)} de {M.shape[1]}")
CODES = M[nodos_completos].apply(lambda s: s.map({"A": 0, "B": 1, "C": 2})).values.astype(int)
def onehot(codes):
    return np.eye(3)[codes].reshape(len(codes), -1)
X_DEC_OH = onehot(CODES)
print("Matriz de decisiones (one-hot):", X_DEC_OH.shape)

**Referencia en unidades originales** (no entran al clustering; sirven para describir los perfiles):

In [ ]:
orig = nodo.groupby("user_id").agg(
    mediana_s=("node_total_duration_ms", lambda s: s.median() / 1000),
    cambios_por_nodo=("cambios", "mean"),
    alternativas_exploradas=("n_exploradas", "mean"),
    pct_eligio_mas_vista=("eligio_mas_vista", "mean"),
    pct_eligio_primera=("eligio_primera", "mean"),
).loc[usuarios]
orig.describe().T[["mean", "std", "min", "50%", "max"]]

## 1. Funciones del modelo

- **Modelo:** clustering jerárquico con enlace de Ward sobre indicadores estandarizados (como indica la propuesta). Se compara entre 2 y 4 grupos.
- **Asignar a personas nuevas:** Ward no tiene `predict`. Se calculan los centroides de los grupos formados con las personas de **entrenamiento**, y cada persona de **prueba** se asigna al centroide más cercano.
- **Control:** KMeans, que sí tiene `predict`, como comprobación de que el resultado no depende del algoritmo.

In [ ]:
KS = [2, 3, 4]
B_NULA, B_BOOT, B_TT, B_NODOS = 1000, 300, 200, 200
rng = np.random.default_rng(SEED)

def ward(X, k):
    return AgglomerativeClustering(n_clusters=k, linkage="ward").fit_predict(X)

def asignar(Xtr, ltr, Xte):
    """Asigna cada fila de Xte al centroide más cercano de los grupos de entrenamiento."""
    cent = np.vstack([Xtr[ltr == j].mean(0) for j in np.unique(ltr)])
    return ((Xte[:, None, :] - cent[None]) ** 2).sum(-1).argmin(1)

def tamanos(lab):
    return "/".join(map(str, sorted(np.bincount(lab), reverse=True)))

def escalar(X, escala):
    return StandardScaler().fit_transform(X) if escala else X

# --- Hipótesis nula: datos sin estructura de grupos -------------------------------------------
def sil_nula_gauss(Xs, k, B):
    """Silhouette de Ward sobre datos normales multivariados con la misma media y correlación."""
    mu, cov = Xs.mean(0), np.cov(Xs.T)
    out = np.empty(B)
    for b in range(B):
        s = rng.multivariate_normal(mu, cov, size=len(Xs), check_valid="ignore", method="eigh")
        s = StandardScaler().fit_transform(s)
        out[b] = silhouette_score(s, ward(s, k))
    return out

def sil_nula_perm(codes, k, B):
    """Silhouette tras permutar cada nodo por separado (rompe el vínculo entre nodos de una misma persona)."""
    out = np.empty(B)
    for b in range(B):
        perm = np.column_stack([rng.permutation(codes[:, j]) for j in range(codes.shape[1])])
        s = onehot(perm)
        out[b] = silhouette_score(s, ward(s, k))
    return out

# --- Estabilidad -------------------------------------------------------------------------------
def estab_bootstrap(Xraw, k, lab_full, escala, B):
    """Entrena con una muestra bootstrap; asigna las personas que no salieron (out-of-bag); compara con los grupos de las 60."""
    n, aris = len(Xraw), []
    for _ in range(B):
        idx = rng.integers(0, n, n)
        oob = np.setdiff1d(np.arange(n), idx)
        sc = StandardScaler().fit(Xraw[idx]) if escala else None
        tr = sc.transform(Xraw[idx]) if escala else Xraw[idx]
        te = sc.transform(Xraw[oob]) if escala else Xraw[oob]
        aris.append(adjusted_rand_score(lab_full[oob], asignar(tr, ward(tr, k), te)))
    return np.array(aris)

def estab_train_test(Xraw, k, lab_full, escala, B):
    """70 % entrenamiento / 30 % prueba. Devuelve dos ARI: contra los grupos que forma la propia muestra de prueba,
    y contra los grupos de las 60 personas."""
    n, vs_propio, vs_full = len(Xraw), [], []
    for _ in range(B):
        tr_i, te_i = train_test_split(np.arange(n), test_size=0.3, random_state=int(rng.integers(1e9)))
        sc = StandardScaler().fit(Xraw[tr_i]) if escala else None
        tr = sc.transform(Xraw[tr_i]) if escala else Xraw[tr_i]
        te = sc.transform(Xraw[te_i]) if escala else Xraw[te_i]
        asign = asignar(tr, ward(tr, k), te)
        vs_propio.append(adjusted_rand_score(asign, ward(te, k)))
        vs_full.append(adjusted_rand_score(lab_full[te_i], asign))
    return np.array(vs_propio), np.array(vs_full)

def estab_nodos(tipo, k, cols, B):
    """Divide los nodos al azar en dos mitades, forma grupos con cada mitad y compara."""
    n_nod = T.shape[1] if tipo == "tel" else CODES.shape[1]
    aris = []
    for _ in range(B):
        p = rng.permutation(n_nod)
        a, b = p[: n_nod // 2], p[n_nod // 2:]
        if tipo == "tel":
            Xa = StandardScaler().fit_transform(features_tel(T, a)[cols].values)
            Xb = StandardScaler().fit_transform(features_tel(T, b)[cols].values)
        else:
            Xa, Xb = onehot(CODES[:, a]), onehot(CODES[:, b])
        aris.append(adjusted_rand_score(ward(Xa, k), ward(Xb, k)))
    return np.array(aris)

# --- Nulos adicionales -------------------------------------------------------------------------
def sil_nula_copula(Xraw, k, B):
    """Como sil_nula_gauss, pero conserva la forma de cada indicador (asimetría) y la correlación entre ellos
    (cópula gaussiana). Los indicadores son asimétricos, así que el nulo normal es algo indulgente."""
    n, d = Xraw.shape
    Z = stats.norm.ppf((stats.rankdata(Xraw, axis=0) / (n + 1)))
    corr = np.corrcoef(Z.T)
    orden = np.sort(Xraw, axis=0)
    out = np.empty(B)
    for b in range(B):
        sim = rng.multivariate_normal(np.zeros(d), corr, size=n, check_valid="ignore", method="eigh")
        u = stats.norm.cdf(sim)
        s = np.column_stack([np.quantile(orden[:, j], u[:, j]) for j in range(d)])
        s = StandardScaler().fit_transform(s)
        out[b] = silhouette_score(s, ward(s, k))
    return out

def estab_nula_gauss(Xs, k, n_sim=30, B=30):
    """Estabilidad (ARI bootstrap y train/test) que se obtiene en datos normales SIN grupos, como línea de base."""
    mu, cov = Xs.mean(0), np.cov(Xs.T)
    b_, t_ = [], []
    for _ in range(n_sim):
        s = rng.multivariate_normal(mu, cov, size=len(Xs), check_valid="ignore", method="eigh")
        lab_s = ward(StandardScaler().fit_transform(s), k)
        b_.append(estab_bootstrap(s, k, lab_s, True, B).mean())
        t_.append(estab_train_test(s, k, lab_s, True, B)[0].mean())
    return float(np.mean(b_)), float(np.mean(t_))

## 2. Evaluación de los grupos

Se evalúan tres conjuntos de variables con 2, 3 y 4 grupos:

1. **Telemetría (3 indicadores):** tiempo típico, cambios entre opciones y alternativas exploradas (los de la propuesta).
2. **Telemetría (8 indicadores):** todos los disponibles.
3. **Decisiones:** las alternativas A/B/C elegidas.

**Cómo leer la tabla:**
- `sil` es el silhouette observado. `nulo_p95` es el percentil 95 del silhouette que da el mismo algoritmo sobre datos **sin grupos reales**. Si `sil` no lo supera, la separación observada se esperaría aun sin estructura (`p_nulo` es la fracción de simulaciones con silhouette ≥ al observado).
- Los **ARI** miden qué tan parecidos son los grupos en dos versiones del análisis (0 = azar, 1 = idénticos):
  - `ARI_boot`: entrenar con una muestra bootstrap y asignar a las personas no incluidas.
  - `ARI_train_test`: entrenar con el 70 % de las personas y asignar al 30 % restante, comparando con los grupos que forma ese 30 % por sí solo.
  - `ARI_test_vs_60`: lo mismo, pero comparando con los grupos de las 60 personas.
  - `ARI_nodos`: grupos formados con una mitad de los nodos contra la otra mitad. Cada mitad usa solo la mitad de la información, así que subestima la estabilidad con todos los nodos.
- `ARI_kmeans`: acuerdo entre Ward y KMeans con los datos completos.

**Líneas de base (qué se obtendría sin grupos reales).**
- `p_copula`: igual que `p_nulo`, pero el nulo conserva la asimetría de cada indicador y su correlación (cópula gaussiana). Los indicadores son asimétricos y el nulo normal es algo indulgente, así que este es el p-valor más exigente.
- `ARI_boot_nulo` y `ARI_tt_nulo`: el ARI de bootstrap y de train/test que da el mismo procedimiento sobre datos normales sin grupos (30 simulaciones). Ward con 2 grupos separa siempre una cola de cualquier nube de puntos, así que un ARI moderado es esperable aun sin estructura: lo que importa es **cuánto supera** a esta línea.
- Con 1.000 simulaciones el menor p posible es 0,001.
- **Selección:** el `k` y el conjunto de indicadores se eligen mirando estos mismos resultados (3 conjuntos × 3 valores de k = 9 combinaciones), así que los p-valores del `k` elegido son optimistas. Una corrección de Bonferroni multiplicaría por 9.

In [ ]:
CONJUNTOS_CL = {
    "Telemetría (3)": dict(tipo="tel", cols=COLS_PROP, escala=True),
    "Telemetría (8)": dict(tipo="tel", cols=COLS_AMPLIA, escala=True),
    "Decisiones": dict(tipo="dec", cols=None, escala=False),
}
def matriz_cl(nombre):
    c = CONJUNTOS_CL[nombre]
    return F_ALL[c["cols"]].values if c["tipo"] == "tel" else X_DEC_OH

LAB, NULOS, ESTAB, filas = {}, {}, {}, []
for nombre, c in CONJUNTOS_CL.items():
    Xraw = matriz_cl(nombre)
    Xs = escalar(Xraw, c["escala"])
    for k in KS:
        lab = ward(Xs, k)
        LAB[(nombre, k)] = lab
        sil = silhouette_score(Xs, lab)
        nul = sil_nula_gauss(Xs, k, B_NULA) if c["tipo"] == "tel" else sil_nula_perm(CODES, k, B_NULA)
        nul_cop = sil_nula_copula(Xraw, k, B_NULA) if c["tipo"] == "tel" else None
        ari_nulo = estab_nula_gauss(Xs, k) if c["tipo"] == "tel" else (np.nan, np.nan)
        boot = estab_bootstrap(Xraw, k, lab, c["escala"], B_BOOT)
        tt_propio, tt_full = estab_train_test(Xraw, k, lab, c["escala"], B_TT)
        nod = estab_nodos(c["tipo"], k, c["cols"], B_NODOS)
        km = KMeans(k, n_init=20, random_state=SEED).fit_predict(Xs)
        NULOS[(nombre, k)] = nul
        ESTAB[(nombre, k)] = dict(boot=boot, tt_propio=tt_propio, tt_full=tt_full, nodos=nod)
        filas.append({"conjunto": nombre, "k": k, "tamaños": tamanos(lab), "sil": sil,
                      "nulo_media": nul.mean(), "nulo_p95": np.percentile(nul, 95),
                      "p_nulo": (np.sum(nul >= sil) + 1) / (len(nul) + 1),
                      "p_copula": (np.sum(nul_cop >= sil) + 1) / (len(nul_cop) + 1) if nul_cop is not None else np.nan,
                      "ARI_boot": boot.mean(), "ARI_boot_nulo": ari_nulo[0],
                      "ARI_train_test": tt_propio.mean(), "ARI_tt_nulo": ari_nulo[1],
                      "ARI_test_vs_60": tt_full.mean(), "ARI_nodos": nod.mean(),
                      "ARI_kmeans": adjusted_rand_score(lab, km)})
tabla_cl = pd.DataFrame(filas)
tabla_cl

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, nombre in zip(axes, CONJUNTOS_CL):
    datos_box = [NULOS[(nombre, k)] for k in KS]
    ax.boxplot(datos_box, positions=range(len(KS)), widths=0.5, showfliers=False,
               boxprops=dict(color="#999999"), medianprops=dict(color="#666666"))
    obs = tabla_cl[tabla_cl.conjunto == nombre]["sil"].values
    ax.scatter(range(len(KS)), obs, color="#C44E52", s=70, zorder=3, label="observado")
    ax.set_xticks(range(len(KS))); ax.set_xticklabels([f"k={k}" for k in KS])
    ax.set_title(nombre); ax.set_ylabel("Silhouette")
axes[0].legend()
plt.suptitle("Silhouette observado (punto) frente a datos sin grupos (cajas)")
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
claves = [("boot", "bootstrap"), ("tt_propio", "train/test"), ("nodos", "mitades de nodos")]
for ax, nombre in zip(axes, CONJUNTOS_CL):
    ancho = 0.25
    for i, (clave, etiqueta) in enumerate(claves):
        medias = [ESTAB[(nombre, k)][clave].mean() for k in KS]
        sds = [ESTAB[(nombre, k)][clave].std() for k in KS]
        ax.bar(np.arange(len(KS)) + (i - 1) * ancho, medias, ancho, yerr=sds, label=etiqueta, capsize=2)
    ax.axhline(0, color="black", lw=0.8)
    ax.set_xticks(range(len(KS))); ax.set_xticklabels([f"k={k}" for k in KS]); ax.set_title(nombre)
axes[0].set_ylabel("ARI (0 = azar, 1 = idéntico)"); axes[0].legend(fontsize=8)
plt.suptitle("Estabilidad de los grupos"); plt.tight_layout(); plt.show()

**Solución de referencia.** Para cada conjunto se elige el `k` en que el silhouette observado más supera el promedio del silhouette bajo la hipótesis nula (así se compensa que un `k` pequeño produce silhouettes altos incluso sin estructura). Esa elección se usa en las secciones siguientes, pero la validación externa de la sección 5 se reporta para todos los `k`.

In [ ]:
tabla_cl["exceso"] = tabla_cl["sil"] - tabla_cl["nulo_media"]
K_REF = tabla_cl.loc[tabla_cl.groupby("conjunto")["exceso"].idxmax()].set_index("conjunto")["k"].to_dict()
print("k de referencia:", K_REF)
tabla_cl.loc[tabla_cl.groupby("conjunto")["exceso"].idxmax(), ["conjunto", "k", "tamaños", "sil", "nulo_p95", "p_nulo", "p_copula", "ARI_boot", "ARI_boot_nulo", "ARI_train_test", "ARI_tt_nulo", "ARI_nodos"]]

## 3. Demostración train/test con una sola partición

Así se usaría el modelo con personas nuevas, por ejemplo en el dashboard: se forman los grupos con el 70 % (entrenamiento) y a cada persona del 30 % restante (prueba) se le asigna el grupo del centroide más cercano. La tabla cruzada compara esa asignación con los grupos que salen al agrupar a las 60 personas juntas.

Con solo 18 personas de prueba el resultado depende mucho de la partición elegida; por eso la sección 2 repite este procedimiento 200 veces.

In [ ]:
nombre = "Telemetría (3)"; K = K_REF[nombre]
Xraw = F_ALL[COLS_PROP].values
tr_i, te_i = train_test_split(np.arange(len(usuarios)), test_size=0.3, random_state=SEED)
sc = StandardScaler().fit(Xraw[tr_i])
Xtr, Xte = sc.transform(Xraw[tr_i]), sc.transform(Xraw[te_i])
ltr = ward(Xtr, K)
asign = asignar(Xtr, ltr, Xte)
lab60 = LAB[(nombre, K)]
print(f"Entrenamiento: {len(tr_i)} personas, grupos de tamaño {tamanos(ltr)} | prueba: {len(te_i)} personas")
print(f"ARI entre la asignación (train) y los grupos de las 60 personas: {adjusted_rand_score(lab60[te_i], asign):.2f}")
pd.crosstab(pd.Series(lab60[te_i], name="grupo con las 60"), pd.Series(asign, name="grupo asignado (train)"))

## 4. Descripción de los perfiles

Con la solución de referencia de los tres indicadores de la propuesta. Los grupos se describen **solo con conducta observable** (más o menos exploración), sin etiquetarlos como "tipos de líder".

In [ ]:
nombre = "Telemetría (3)"; K = K_REF[nombre]; lab = LAB[(nombre, K)]
perfil = orig.groupby(lab).mean()
perfil.insert(0, "n", np.bincount(lab))
print(f"Perfiles de {nombre}, k = {K} (promedios en unidades originales)")
display(perfil)

zmed = pd.DataFrame(StandardScaler().fit_transform(F_ALL[COLS_AMPLIA]), index=usuarios, columns=COLS_AMPLIA).groupby(lab).mean()
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
sns.heatmap(zmed.T, annot=True, fmt=".2f", cmap="RdBu_r", center=0, vmin=-1.2, vmax=1.2, ax=axes[0],
            cbar_kws={"label": "promedio del grupo (z entre personas)"})
axes[0].set_title("Perfil de cada grupo (indicadores de telemetría)"); axes[0].set_xlabel("grupo")

pca2 = PCA(2).fit(StandardScaler().fit_transform(F_ALL[COLS_AMPLIA]))
pts = pca2.transform(StandardScaler().fit_transform(F_ALL[COLS_AMPLIA]))
for g in np.unique(lab):
    axes[1].scatter(pts[lab == g, 0], pts[lab == g, 1], label=f"grupo {g} (n={np.sum(lab == g)})", s=45)
axes[1].set_xlabel(f"PC1 ({pca2.explained_variance_ratio_[0]:.0%})"); axes[1].set_ylabel(f"PC2 ({pca2.explained_variance_ratio_[1]:.0%})")
axes[1].set_title("Personas en el plano de los dos primeros componentes"); axes[1].legend()
plt.tight_layout(); plt.show()

# ¿Un grupo está formado por personas extremas?
ext_z = np.abs(StandardScaler().fit_transform(F_ALL[COLS_PROP])).max(axis=1)
print("Personas con algún indicador a más de 2,5 desviaciones estándar por grupo:",
      pd.Series(ext_z > 2.5).groupby(lab).sum().to_dict())

## 5. Validación externa: ¿los grupos se relacionan con el MLQ?

Se compara el MLQ externo entre grupos con la prueba de **Kruskal-Wallis** (no asume normalidad; con 2 grupos equivale a Mann-Whitney). El tamaño del efecto es ε² = H / (n − 1), que va de 0 a 1.

Se hacen 12 comparaciones por solución de grupos, así que se esperan **0,6 resultados con p < 0,05 solo por azar**. Dentro de cada solución se corrige con FDR (Benjamini-Hochberg).

También se comparan los grupos con las decisiones de COMPASS (`dec_*`): ayuda a ver si el perfil de cursor se asocia con *qué* eligen.

In [ ]:
def kruskal_tab(lab, df, cols):
    filas = []
    for c in cols:
        grupos = [df[c].values[lab == j] for j in np.unique(lab)]
        H, p = stats.kruskal(*grupos)
        filas.append({"variable": c, "H": H, "p": p, "epsilon2": H / (len(lab) - 1)})
    t = pd.DataFrame(filas)
    t["p_fdr"] = multipletests(t["p"], method="fdr_bh")[1]
    return t

Y_u = Y.loc[usuarios]; Xd_u = X_dec.loc[usuarios]
filas = []
for (nombre, k), lab in LAB.items():
    for etiqueta, df, cols in [("MLQ externo", Y_u, OBJ_MLQ), ("Decisiones COMPASS", Xd_u, list(Xd_u.columns))]:
        t = kruskal_tab(lab, df, cols)
        mejor = t.loc[t["p"].idxmin()]
        filas.append({"conjunto": nombre, "k": k, "contra": etiqueta, "n_pruebas": len(t),
                      "p<0,05 (esperadas por azar)": f"{(t.p < 0.05).sum()} ({0.05 * len(t):.1f})",
                      "p_min": mejor["p"], "p_min_fdr": t["p_fdr"].min(),
                      "variable_p_min": mejor["variable"], "epsilon2_max": t["epsilon2"].max()})
valid_ext = pd.DataFrame(filas)
valid_ext

In [ ]:
nombre = "Telemetría (3)"; K = K_REF[nombre]; lab = LAB[(nombre, K)]
t_ref = kruskal_tab(lab, Y_u, OBJ_MLQ)
print(f"Detalle: {nombre}, k = {K}, MLQ externo")
display(t_ref.sort_values("p"))

medias = Y_u[OBJ_MLQ].groupby(lab).mean()
zmlq = (medias - Y_u[OBJ_MLQ].mean()) / Y_u[OBJ_MLQ].std()
fig, ax = plt.subplots(figsize=(10, 3 + 0.4 * K))
sns.heatmap(zmlq, annot=True, fmt=".2f", cmap="RdBu_r", center=0, vmin=-1, vmax=1, ax=ax,
            cbar_kws={"label": "MLQ promedio del grupo (z)"})
ax.set_ylabel("grupo"); ax.set_title(f"MLQ externo promedio por grupo de telemetría ({nombre}, k={K})")
plt.tight_layout(); plt.show()

### 5b. Detalle frente a las decisiones de COMPASS y controles

Si los grupos de cursor difieren en las decisiones, hay que descartar explicaciones triviales antes de interpretarlo:

- **Efecto de la aplicación.** La sesión se hizo en 11 días distintos. Si un grupo se concentra en ciertos días u horas, la diferencia podría deberse a la logística o a un cambio en la plataforma y no a la conducta de las personas. Se compara fecha, hora y duración total de la sesión entre grupos.
- **Personas extremas.** Se repite el clustering sin las personas con algún indicador a más de 2,5 desviaciones estándar, para ver si el grupo pequeño depende de ellas.

In [ ]:
nombre = "Telemetría (3)"; K = K_REF[nombre]; lab = LAB[(nombre, K)]
t_dec = kruskal_tab(lab, Xd_u, list(Xd_u.columns))
med_dec = Xd_u.groupby(lab).mean().T
med_dec.columns = [f"media grupo {g}" for g in med_dec.columns]
print(f"Decisiones COMPASS (promedio por decisión) según grupo de telemetría ({nombre}, k = {K})")
display(t_dec.set_index("variable").join(med_dec).sort_values("p"))

ses = pd.read_csv(DATA / "sessions.csv").set_index("user_id").loc[usuarios]
ini = pd.to_datetime(ses["start_time"], utc=True).dt.tz_convert("America/Santiago")
ses_df = pd.DataFrame({
    "dia_aplicacion": (ini.dt.normalize() - ini.dt.normalize().min()).dt.days,
    "hora_inicio": ini.dt.hour + ini.dt.minute / 60,
    "duracion_sesion_min": (pd.to_datetime(ses["end_time"], utc=True) - pd.to_datetime(ses["start_time"], utc=True)).dt.total_seconds() / 60,
}, index=usuarios)
print("\nControl de aplicación: ¿los grupos difieren en cuándo y cuánto duró la sesión?")
display(kruskal_tab(lab, ses_df, list(ses_df.columns)).set_index("variable"))
print("Personas por día de aplicación y grupo:")
display(pd.crosstab(ini.dt.date.values, pd.Series(lab, name="grupo")))
rho_dia, p_dia = stats.spearmanr(ses_df["dia_aplicacion"], F_ALL["tel_n_exploradas"])
print(f"Correlación día de aplicación vs alternativas exploradas: ρ = {rho_dia:.2f} (p = {p_dia:.3f})")

# Prueba exacta de que el grupo pequeño falta en los últimos días (día 14 en adelante = 17-19 de junio).
# El corte se eligió después de mirar la tabla, así que es descriptivo.
g_peq = pd.Series(lab).value_counts().idxmin()
tarde = (ses_df["dia_aplicacion"] >= 14).values
tab_f = [[int(np.sum((lab == g_peq) & tarde)), int(np.sum((lab != g_peq) & tarde))],
         [int(np.sum((lab == g_peq) & ~tarde)), int(np.sum((lab != g_peq) & ~tarde))]]
print(f"Grupo pequeño / resto, días 17-19: {tab_f[0]}; antes: {tab_f[1]} | Fisher exacto p = {stats.fisher_exact(tab_f)[1]:.4f}")

# Control de fecha más directo: ¿siguen difiriendo las decisiones entre grupos si se compara solo DENTRO de cada día?
# Se permutan las etiquetas de grupo dentro de cada día de aplicación (11 estratos).
def perm_estratificada(lab, y, estratos, B=5000):
    r = stats.rankdata(y)
    obs = abs(r[lab == g_peq].mean() - r[lab != g_peq].mean())
    cnt = 0
    for _ in range(B):
        l2 = lab.copy()
        for e in np.unique(estratos):
            idx = np.where(estratos == e)[0]
            l2[idx] = rng.permutation(lab[idx])
        cnt += abs(r[l2 == g_peq].mean() - r[l2 != g_peq].mean()) >= obs
    return (cnt + 1) / (B + 1)

estr = ses_df["dia_aplicacion"].values
perm_dia = pd.DataFrame({"variable": Xd_u.columns,
                         "p_permutacion_dentro_de_dia": [perm_estratificada(lab, Xd_u[c].values, estr) for c in Xd_u.columns]})
perm_dia["p_fdr"] = multipletests(perm_dia["p_permutacion_dentro_de_dia"], method="fdr_bh")[1]
print("\nDecisiones COMPASS entre grupos, comparando solo dentro de cada día de aplicación:")
display(perm_dia.set_index("variable").sort_values("p_permutacion_dentro_de_dia"))

# ¿El día de aplicación también se asocia con las decisiones? Si sí, la diferencia entre grupos podría ser un efecto de fecha.
filas = []
for y in Xd_u.columns:
    r, p = stats.spearmanr(ses_df["dia_aplicacion"], Xd_u[y])
    filas.append({"variable": y, "rho_con_dia": r, "p": p})
print("\nDía de aplicación vs decisiones de COMPASS:")
display(pd.DataFrame(filas).set_index("variable"))

# Comparación restringida a la ventana de fechas en que hay personas de ambos grupos
ult_dia_grupo_pequeno = ses_df["dia_aplicacion"][lab == pd.Series(lab).value_counts().idxmin()].max()
ventana = (ses_df["dia_aplicacion"] <= ult_dia_grupo_pequeno).values
print(f"\nSolo personas hasta el día {ult_dia_grupo_pequeno} de aplicación: {ventana.sum()} personas, "
      f"grupos {np.bincount(lab[ventana]).tolist()}")
display(kruskal_tab(lab[ventana], Xd_u[ventana], list(Xd_u.columns)).set_index("variable").sort_values("p"))

# Sensibilidad: sin las personas extremas
mask = ext_z <= 2.5
lab_m = ward(StandardScaler().fit_transform(F_ALL[COLS_PROP].values[mask]), K)
print(f"\nSin las {np.sum(~mask)} personas extremas: grupos {tamanos(lab_m)}; "
      f"ARI contra la solución con las 60 = {adjusted_rand_score(lab[mask], lab_m):.2f}")

## 6. Clustering por decisiones: ¿qué eligen?

Agrupa a las personas por **qué alternativa eligieron** en los 24 nodos que todos recorrieron. Se comprueba además si estos grupos coinciden con los grupos de telemetría (¿quienes eligen parecido también exploran parecido?). El p-valor del ARI sale de permutar las etiquetas.

In [ ]:
K_dec = K_REF["Decisiones"]; lab_dec = LAB[("Decisiones", K_dec)]
print(f"Decisiones, k = {K_dec}, tamaños {tamanos(lab_dec)}")
print("Atención: estos grupos NO superan el nulo (sección 2); las tablas siguientes son solo descriptivas y no deben leerse como perfiles reales.")

def cramers_v(a, b):
    ct = pd.crosstab(a, b)
    chi2 = stats.chi2_contingency(ct, correction=False)[0]
    return np.sqrt(chi2 / (ct.values.sum() * (min(ct.shape) - 1)))

V = pd.Series({n: cramers_v(lab_dec, CODES[:, j]) for j, n in enumerate(nodos_completos)}).sort_values(ascending=False)
print("\nNodos que más separan los grupos (V de Cramér):")
display(V.head(6).to_frame("V"))
for n in V.index[:3]:
    print(f"\n{n}: % que elige cada alternativa dentro de cada grupo")
    display(pd.crosstab(pd.Series(lab_dec, name="grupo"), M.loc[usuarios, n].values, normalize="index").round(2))

filas = []
for k in KS:
    for nombre in ["Telemetría (3)", "Telemetría (8)"]:
        a, b = LAB[(nombre, k)], LAB[("Decisiones", k)]
        ari = adjusted_rand_score(a, b)
        nul = np.array([adjusted_rand_score(a, rng.permutation(b)) for _ in range(2000)])
        filas.append({"k": k, "telemetría": nombre, "ARI": ari, "p_perm": (np.sum(nul >= ari) + 1) / (len(nul) + 1)})
print("\nAcuerdo entre grupos por telemetría y por decisiones:")
pd.DataFrame(filas)

## 7. Alternativa: un índice continuo en vez de grupos

Si la conducta del cursor forma un **continuo** (más o menos exploración) y no tipos separados, un puntaje continuo la describe mejor que un grupo. Se resume la telemetría en componentes principales (PCA), se orienta el primero para que valores altos signifiquen más exploración y se mide su relación con el MLQ (Spearman, corrección FDR).

Es una descripción de las 60 personas, no un modelo predictivo.

In [ ]:
sc_pca = StandardScaler().fit(F_ALL[COLS_AMPLIA])
pca = PCA().fit(sc_pca.transform(F_ALL[COLS_AMPLIA]))
scores = pca.transform(sc_pca.transform(F_ALL[COLS_AMPLIA]))
carga = pd.DataFrame(pca.components_[:3].T, index=COLS_AMPLIA, columns=["PC1", "PC2", "PC3"])
for j, c in enumerate(["PC1", "PC2", "PC3"]):          # orientar: más cambios entre opciones = valor alto
    if carga.loc["tel_cambios", c] < 0:
        carga[c] *= -1; scores[:, j] *= -1
print("Varianza explicada:", (pca.explained_variance_ratio_[:3] * 100).round(1), "%")
display(carga)

indice = pd.DataFrame(scores[:, :2], index=usuarios, columns=["pc1_exploracion", "pc2"])
indice["pct_exploracion"] = indice["pc1_exploracion"].rank(pct=True) * 100

filas = []
for pc in ["pc1_exploracion", "pc2"]:
    for y in OBJ_MLQ:
        r, p = stats.spearmanr(indice[pc], Y_u[y])
        filas.append({"componente": pc, "objetivo": y, "rho": r, "p": p})
corr_pc = pd.DataFrame(filas)
corr_pc["p_fdr"] = multipletests(corr_pc["p"], method="fdr_bh")[1]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(indice["pc1_exploracion"], bins=18, color="#4C72B0", edgecolor="white")
axes[0].set_xlabel("PC1: índice de exploración"); axes[0].set_title("Distribución del índice (¿un continuo o dos bultos?)")
mat = corr_pc.pivot(index="componente", columns="objetivo", values="rho")[OBJ_MLQ]
sns.heatmap(mat, annot=True, fmt=".2f", cmap="RdBu_r", center=0, vmin=-0.5, vmax=0.5, ax=axes[1],
            annot_kws={"size": 7}, cbar_kws={"label": "ρ de Spearman"})
axes[1].set_title("Índice continuo vs MLQ externo")
plt.tight_layout(); plt.show()
print("Índice continuo de exploración vs MLQ externo (más fuertes):")
display(corr_pc.sort_values("p").head(6))

# También contra las decisiones de COMPASS
filas = []
for y in Xd_u.columns:
    r, p = stats.spearmanr(indice["pc1_exploracion"], Xd_u[y])
    filas.append({"variable": y, "rho": r, "p": p})
corr_dec = pd.DataFrame(filas)
corr_dec["p_fdr"] = multipletests(corr_dec["p"], method="fdr_bh")[1]
print("Índice continuo de exploración vs decisiones de COMPASS:")
corr_dec.sort_values("p")

## 8. Guardar resultados para el dashboard

Una fila por persona: grupo de referencia de cada solución, índice continuo y percentil de exploración.

In [ ]:
OUT.mkdir(exist_ok=True)
salida = pd.DataFrame(index=pd.Index(usuarios, name="user_id"))
for nombre in CONJUNTOS_CL:
    salida[f"grupo_{nombre}_k{K_REF[nombre]}"] = LAB[(nombre, K_REF[nombre])]
salida = salida.join(indice).join(orig)
salida.to_csv(OUT / "perfiles_cursor.csv")
print("Guardado en", (OUT / "perfiles_cursor.csv").resolve())
salida.head()

## 9. Conclusiones (ejecución del 26-11-2026, semilla 42)

**1. Qué se hizo.** Clustering jerárquico (Ward), aprendizaje no supervisado. No hay variable que predecir; los grupos se validaron contra datos sin estructura, con estabilidad train/test, bootstrap y mitades de nodos, y contra variables externas. Todo es exploratorio.

**2. Por decisiones (qué eligen) no hay grupos.** El silhouette de las 24 decisiones comunes es ≈ 0,03, igual al de datos permutados (p entre 0,77 y 0,92), y el ARI entre mitades de nodos es ≈ 0. Las tablas de la sección 6 son solo descriptivas.

**3. Por telemetría (cómo eligen) no aparecen tipos claros: aparece una cola de poca exploración.**
- Con 3 o 4 grupos no hay estructura (p ≥ 0,68).
- Con 2 grupos (51 y 9 personas) el silhouette supera al de datos **normales** sin grupos (3 indicadores: p = 0,001; 8 indicadores: p = 0,003), pero no con un nulo que **respeta la asimetría** de los indicadores: p ≈ 0,05 en ambos casos. Tres implementaciones distintas de ese nulo dieron entre 0,03 y 0,07. Además, el 2 se eligió entre 9 combinaciones (3 conjuntos × 3 valores de k); con una corrección de Bonferroni (× 9) ya no es significativo.
- Lectura: el grupo pequeño es mejor descrito como la **cola izquierda de una distribución asimétrica** (el histograma de PC1 muestra un bloque grande y una cola) que como un tipo de usuario separado.
- Esas 9 personas (15 %) exploran y cambian mucho menos: en promedio, la mediana individual por nodo es de 13,1 s frente a 19,7 s, hacen 2,2 cambios entre opciones frente a 4,5 y exploran 2,3 alternativas frente a 2,8.

**4. Estabilidad: modesta frente a la línea de base.** Ward con 2 grupos separa siempre una cola, así que se compara con datos sin grupos (columnas `*_nulo`).
- **Tres indicadores:** bootstrap 0,79 contra 0,57 sin grupos; train/test 0,55 contra 0,46. Mitades de nodos: 0,38 (sin línea de base, y subestima la estabilidad con todos los nodos). El 0,87 "contra las 60" es optimista, porque esas 60 incluyen a las personas de prueba.
- **Ocho indicadores:** bootstrap 0,61 contra 0,43; train/test 0,25 contra 0,34, es decir, no distinguible de datos sin grupos.
- KMeans coincide con Ward (ARI = 0,92) y el resultado se mantiene sin las 3 personas más extremas (grupos 50/7, ARI = 1,00).

**5. No hay evidencia de que los grupos se relacionen con el MLQ externo.** Con 2 grupos el p mínimo entre las 12 variables es 0,13. Con otras soluciones aparece algún p nominal bajo (por ejemplo, `mi` con p = 0,039 para ocho indicadores y 3 grupos), pero ninguno sobrevive a la corrección (p_FDR ≥ 0,35). El índice continuo da `mi` (ρ = −0,28, p = 0,03) y `rc` (ρ = −0,26, p = 0,049), también sin sobrevivir (p_FDR ≥ 0,58). Con un grupo de 9 personas la prueba tiene poca potencia: es ausencia de evidencia, no evidencia de ausencia.

**6. Sí hay diferencias en las decisiones de COMPASS.** Las 9 personas de poca exploración eligieron alternativas menos transformacionales y más cercanas a laissez-faire:

| Puntaje por decisión | Grupo grande | Grupo pequeño | p_FDR (Kruskal) | p_FDR (permutación dentro de cada día) |
|---|---|---|---|---|
| `dec_iic` | 2,00 | 1,67 | 0,002 | 0,002 |
| `dec_iia` | 2,08 | 1,74 | 0,002 | 0,002 |
| `dec_ei` | 0,82 | 0,62 | 0,002 | 0,006 |
| `dec_ci` | 1,24 | 1,02 | 0,007 | 0,030 |
| `dec_lf` | 0,26 | 0,40 | 0,011 | 0,016 |

- **Control de fecha.** La permutación dentro de cada día mantiene las cinco diferencias. Comparando solo a las 35 personas de la misma ventana de fechas, `dec_iic`, `dec_iia` y `dec_ei` se mantienen (p_FDR = 0,002), `dec_lf` se debilita (0,036) y `dec_ci` queda en el límite (0,051).
- **Con el índice continuo la relación es débil y no significativa** (`dec_iia`: ρ = 0,23, p_FDR = 0,53). Con este tamaño de muestra no se puede distinguir un efecto concentrado en la cola de la falta de potencia.

**7. Cautelas.**
- **Grupo pequeño.** Son 9 personas, así que cualquier comparación tiene poca potencia.
- **Efecto de aplicación.** El grupo falta en los últimos días: ninguna de las 25 personas de los días 17 a 19 de junio pertenece a él (Fisher exacto, p = 0,0075; el corte se eligió después de mirar la tabla). A mayor día de aplicación hay más alternativas exploradas (ρ = 0,29, p = 0,026), y el día también se asocia débilmente con `dec_iia` (ρ = 0,26, p = 0,045) y `dec_ci` (ρ = 0,25, p = 0,058). La permutación dentro de cada día reduce el riesgo, pero con estos datos no se puede descartar del todo un efecto de fecha (cambio en la plataforma, en la forma de aplicar o en la muestra). Conviene confirmarlo con quien aplicó el piloto.
- **Elección de k y de indicadores.** Se hizo mirando estos mismos resultados; los p-valores del `k` elegido son optimistas.
- **Las decisiones no son una medida independiente.** `dec_*` se calcula con las etiquetas expertas de las alternativas elegidas.
- **Muchas pruebas.** Se corrige con FDR dentro de cada solución, no entre soluciones.

**8. Lectura para el proyecto.**
- La conducta del cursor no forma "tipos de usuario" claros. Lo defendible es que la exploración es un **continuo asimétrico**, con una cola pequeña de personas que exploran y cambian mucho menos y que, además, eligieron alternativas menos transformacionales (con las cautelas del punto 7).
- Para el dashboard conviene mostrar el **índice continuo y su percentil**, y señalar a quienes caen en la cola, sin presentarlo como un tipo de líder.
- Ningún resultado valida estos grupos contra el MLQ externo.